# Measure objects and crop images

**What it does.** Measure every masked object and optionally write a cropped image per object.

**When to use it.** Straight after masking. The crops it writes are the input to every image classifier in spaCR.

**What you get.** `measurements/measurements.db` and, when cropping is on, one image per object under `data/`.

---

> Every path below is a placeholder. Point `src` at your own data before running.
> Nothing in this notebook writes outside the folder you give it.

## 1. Check the install

If this cell fails, the rest cannot work. It reports the version and whether a GPU is visible — segmentation and training are usable on CPU but slow.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. The function this notebook runs

`spacr.measure.measure_crop`

```
measure_crop(settings)
```

Extract per-object morphology/intensity measurements and (optionally) cropped PNGs from mask stacks.

In [ ]:
from spacr.measure import measure_crop

## 3. Settings

`spacr.settings.get_measure_crop_settings` fills in every default, so you only have to write down what differs. The cell below prints the full set as it exists in this version — treat that output as the reference, not this notebook.

Change values in `settings`, not in the defaults helper.

In [ ]:
from spacr.settings import get_measure_crop_settings

defaults = get_measure_crop_settings({})
for key in sorted(defaults):
    print(f'{key:38s} {defaults[key]!r}')

### Every setting this function accepts

Every key, with its default and what it controls. Edit values in place; delete nothing — a key left at its default behaves exactly as if it were absent.

Generated from this installed version by `tools/build_notebook_settings.py`, so these are the real keys, the real defaults and the real descriptions. Re-run that tool after upgrading spaCR.

In [ ]:
# Generated by tools/build_notebook_settings.py — do not edit by hand.
# Values are this installed version's defaults; the text above each key is
# its live description. Edit values in place -- a key left at its default
# behaves exactly as if it were absent.

# spacr.measure.measure_crop  (55 settings)
settings = {
    # (float or None) - The ratio of z step to xy pixel size (dz / dxy),
    # which is how volumetric mode knows how far apart two planes really
    # are. Left at 1.0 on a confocal stack, where the z step is routinely
    # 3-10x the xy pixel, the segmenter reads a 5 um gap as a 5 pixel gap
    # and fuses everything along z into columns. spaCR will not guess:
    # leave this None and set voxel_size_z_um / voxel_size_xy_um instead
    # and it is derived, and if neither is known a volumetric run STOPS
    # rather than silently assuming 1.0. Measure reads it too, for 3-D
    # regionprops and distance transforms. Default None.
    'anisotropy': None,

    # (bool) - For every pair of measured channels and every object mask,
    # compute a per-object Pearson correlation plus Manders M1/M2 at each
    # cut-off in manders_thresholds, stored as
    # <object>_channel_i_channel_j_* columns. Needs at least two channels.
    # Turn it off to cut measurement time and database size when
    # colocalisation is not part of the phenotype. Default True.
    'calculate_correlation': True,

    # (int) - Position along the last axis of each merged/*.npy array
    # where the cell label mask sits. Merged arrays are ordered [image
    # channels..., cell, nucleus, pathogen, organelle], so the default 4
    # assumes the four channels 0-3 were kept; keep fewer channels and
    # every mask dim shifts down. None makes measure_crop skip all cell
    # measurements and cell crops. Default 4.
    'cell_mask_dim': 4,

    # (int) - (Depreceated) Pixel-area floor applied to cell labels during
    # measurement: any cell smaller than this is erased from the mask
    # before features are extracted. Superseded by cell_min_area, which
    # filters at segmentation time, but this one still runs if you set it.
    # 0 or None disables it. Default 0.
    'cell_min_size': 0,

    # (list of int) - Zero-indexed image channels kept in merged/*.npy and
    # measured by measure_crop; each entry produces its own
    # <object>_channel_<n>_* intensity columns. The list length fixes
    # where masks land, so cell/nucleus/pathogen_mask_dim must shift if
    # you change it. Preprocessing silently resets it to range(n) when it
    # does not match the number of channel folders found. Default
    # [0,1,2,3].
    'channels': [0, 1, 2, 3],

    # (bool) - Add the CORRECT Manders coefficients (manders_m1,
    # manders_m2, manders_overlap_coefficient) alongside the existing
    # M1_correlation_* columns, which are not Manders' coefficients
    # despite their name and are deprecated. The old columns are left
    # untouched so plates measured before and after this setting still
    # agree; read the new ones. Default False.
    'corrected_manders': False,

    # (list) - Which mask each PNG crop is centred on: any of 'cell',
    # 'nucleus', 'pathogen', 'cytoplasm' or 'organelle'. One crop set is
    # written per entry into <mode>_png/ folders, so ['cell','nucleus']
    # doubles the images written and the rows added to png_list. A single
    # png_size such as [224,224] is broadcast to every mode, and so are
    # dialate_pngs and dialate_png_ratios - pass a list only when the
    # modes need different values. A list shorter than crop_mode reuses
    # its last entry for the rest and says so. Default ['cell'].
    'crop_mode': ['cell'],

    # (bool) - Derive a cytoplasm object per cell (cell mask with nucleus,
    # pathogen and organelle pixels removed) and write it to its own
    # cytoplasm table, which recruitment ratios such as pathogen/cytoplasm
    # intensity are computed from. Requires a cell mask; measure_crop
    # switches it on automatically whenever cell_mask_dim is set, so the
    # value you enter is usually overridden. Default False.
    'cytoplasm': False,

    # (int) - (Depreceated) Pixel-area floor for the cytoplasm mask, which
    # is the cell mask with nucleus, pathogen and organelle pixels
    # removed. Cytoplasm regions below this are erased before measurement,
    # so their host cell yields no cytoplasm features and any recruitment
    # ratio built on them is lost. 0 or None disables. Default 0.
    'cytoplasm_min_size': 0,

    # (list of float) - Dilation amount as a fraction of object size: the
    # mask is grown by ratio * sqrt(object area) pixels of binary
    # dilation, so 0.2 expands a cell by roughly 20% of its diameter and
    # pulls in surrounding background. Only used when dialate_pngs is
    # True. A single value applies to every crop_mode entry; pass a list
    # only when the modes need different ratios. Default [0.2].
    'dialate_png_ratios': [0.2],

    # (bool) - Grow each object mask before cropping so the PNG keeps a
    # rim of surrounding pixels instead of a hard mask edge; the amount
    # comes from dialate_png_ratios. May be a list with one value per
    # crop_mode entry (a single value applies to all of them), and is
    # forced off for crop_mode 'cytoplasm'. Enable when context around the
    # object helps the classifier. Default False.
    'dialate_pngs': False,

    # (int or None) - Sigma in pixels of the Gaussian blur applied to each
    # channel before measuring intensity-weighted centroid distances from
    # cells to nuclei and pathogens. Larger values smooth out speckle so
    # the weighted centroid follows broad signal. None or 0 skips these
    # distance features entirely. Needs a cell mask plus a nucleus or
    # pathogen mask. Default 10.
    'distance_gaussian_sigma': 10,

    # (bool) - Validate the settings against the data they point at, print
    # what the run WOULD do, and stop before any compute. Checks that src
    # exists and holds the expected files, that every channel and mask-
    # plane index is inside the number of planes actually present, and
    # that the models, barcode CSVs or measurements.db the run needs are
    # on disk -- each problem printed with a suggested fix, then a plan of
    # what would be segmented, measured and written where. Nothing is
    # written, no model is loaded and the GPU is never touched, so a
    # settings mistake costs seconds instead of a whole run. Default
    # False.
    'dry_run': False,

    # (str) - Free-text run label. Its real effect is naming the exported
    # PNG dataset tar as <YYMMDD>_<experiment>.tar (a random-numbered
    # variant is used if that name already exists), so give each screen a
    # distinct value to avoid confusing dataset tars. It is also passed to
    # the measurement-database writer but not stored there. Defaults vary
    # by pipeline: 'exp', 'exp.' or 'experiment_1'.
    'experiment': 'exp',

    # (bool) - Compute grey-level co-occurrence-matrix homogeneity for
    # every object in every channel, adding one homogeneity_distance_<d>
    # column per entry in homogeneity_distances. Homogeneity is high for
    # smooth, evenly filled objects and low for punctate or grainy ones,
    # so keep it on for texture phenotypes; disabling it noticeably speeds
    # up measurement. Default True.
    'homogeneity': True,

    # (list) - Pixel offsets used to build each object's grey-level co-
    # occurrence matrix; every entry adds one homogeneity_distance_<d>
    # feature per channel. Small offsets capture fine-grained texture,
    # large ones capture coarse structure, and offsets larger than the
    # object itself carry no signal. More entries means more features and
    # slower measurement. Default [8, 16, 32].
    'homogeneity_distances': [8, 16, 32],

    # (list) - Percentiles (0-100) at which Manders' overlap coefficients
    # are computed. For each object, each entry thresholds both channels
    # at that percentile; pixels above both count as overlap, and M1/M2
    # report each channel's fraction of total object intensity there,
    # saved as M1_correlation_<t> and M2_correlation_<t>. High values
    # isolate the brightest puncta. Requires calculate_correlation.
    # Default [15, 85, 95].
    'manders_thresholds': [15, 85, 95],

    # (float or None) - Fraction of failed items above which the run
    # aborts rather than finishing and reporting. 0.2 means 'stop once
    # more than a fifth of the fields have failed', on the grounds that
    # whatever is left is no longer the experiment. The ledger is stamped
    # into the artifact before the abort, so the evidence survives. None
    # (the default) never aborts on rate alone - every failure is still
    # counted and reported, and the artifact is still marked partial.
    # Default None.
    'max_failure_rate': None,

    # (bool) - During measurement, reconcile pathogens straddling two
    # host-cell masks: if 90 percent or more of the pathogen lies in one
    # cell, its pixels in the neighbours are erased; otherwise the
    # overlapping cell labels are fused into a single cell. Switch off to
    # keep the raw cell segmentation when parasites legitimately touch two
    # cells. Default True.
    'merge_edge_pathogen_cells': True,

    # (int) - CPU workers for parallel stages: measurement, mask
    # adjustment, DataLoader loading, and the sklearn/UMAP calls where -1
    # means every core. Raise it to shorten CPU-bound steps until RAM or
    # disk I/O saturates. Note the measure-and-crop pipeline overrides
    # your value with cpu_count()-4. Defaults vary by pipeline:
    # cpu_count()-4, -1, or None.
    'n_jobs': max(1, (__import__('os').cpu_count() or 1) - 2),

    # (bool) - Percentile-normalize each image channel (2nd to 98th
    # percentile, clipped to 0-1) before display or model input; in the
    # activation-map tool this rescales the image the CAM/saliency heatmap
    # is drawn over. Turn it on when raw channels are too dim to read
    # under the overlay. Affects display and input scaling only, never
    # stored pixels. Default True.
    'normalize': False,

    # (str) - Percentile source used to rescale cropped PNGs, and only
    # active when 'normalize' is a [low, high] percentile pair: 'png'
    # stretches each crop to its own percentiles, maximising per-object
    # contrast; 'fov' uses percentiles from the whole field, keeping
    # brightness comparable between objects. Choose 'fov' if crop
    # intensities will be compared. Default 'png'.
    'normalize_by': 'png',

    # (int) - Position along the last axis of each merged/*.npy array
    # where the nucleus label mask sits, one plane after the cell mask.
    # With the default four image channels (0-3) that is 5; keep a
    # different number of channels and it shifts by the same amount. None
    # makes measure_crop skip nucleus measurements and cell-to-nucleus
    # linking. Default 5.
    'nucleus_mask_dim': 5,

    # (int) - (Depreceated) Minimum nucleus size in pixels^2 applied
    # during measure_crop: labels covering fewer pixels than this are
    # erased from the nucleus mask before any feature is measured, so
    # those nuclei never reach the database. 0 (default) disables it.
    # Prefer nucleus_min_area, which filters at segmentation time.
    'nucleus_min_size': 0,

    # (int) - Position along the last axis of each merged/*.npy array
    # where the organelle label mask sits. Masks follow the image channels
    # in the order cell, nucleus, pathogen, organelle, so with four
    # channels and all three other masks present it is 7. Leave it
    # unset/None and organelles are not measured at all. No default is
    # applied.
    'organelle_mask_dim': None,

    # (int) - (Depreceated) Minimum object area in square pixels. Most
    # classical segmenters and the U-Net discard smaller components during
    # segmentation via remove_small_objects (the LoG/DoG spot methods do
    # not, and the ring method applies a quarter of it, floor 3, to its
    # edge image), and the value is always applied again to the finished
    # label image. Despite the marker it is still live - raise it to clear
    # dim specks and hot pixels, lower it to keep faint puncta. Default
    # 10; 0 disables.
    'organelle_min_size': 0,

    # (int) - Position along the last axis of each merged/*.npy array
    # where the organelle 2 label mask sits. Masks follow the image
    # channels in the order cell, nucleus, pathogen, organelle, so with
    # four channels and all three other masks present it is 7. Leave it
    # unset/None and organelles are not measured at all. No default is
    # applied.
    'organelleb_mask_dim': None,

    # (int) - (Depreceated) Minimum object area in square pixels. Most
    # classical segmenters and the U-Net discard smaller components during
    # segmentation via remove_small_objects (the LoG/DoG spot methods do
    # not, and the ring method applies a quarter of it, floor 3, to its
    # edge image), and the value is always applied again to the finished
    # label image. Despite the marker it is still live - raise it to clear
    # dim specks and hot pixels, lower it to keep faint puncta. Default
    # 10; 0 disables.
    'organelleb_min_size': 0,

    # (int) - Position along the last axis of each merged/*.npy array
    # where the organelle 3 label mask sits. Masks follow the image
    # channels in the order cell, nucleus, pathogen, organelle, so with
    # four channels and all three other masks present it is 7. Leave it
    # unset/None and organelles are not measured at all. No default is
    # applied.
    'organellec_mask_dim': None,

    # (int) - (Depreceated) Minimum object area in square pixels. Most
    # classical segmenters and the U-Net discard smaller components during
    # segmentation via remove_small_objects (the LoG/DoG spot methods do
    # not, and the ring method applies a quarter of it, floor 3, to its
    # edge image), and the value is always applied again to the finished
    # label image. Despite the marker it is still live - raise it to clear
    # dim specks and hot pixels, lower it to keep faint puncta. Default
    # 10; 0 disables.
    'organellec_min_size': 0,

    # (int) - Position along the last axis of each merged/*.npy array
    # where the organelle 4 label mask sits. Masks follow the image
    # channels in the order cell, nucleus, pathogen, organelle, so with
    # four channels and all three other masks present it is 7. Leave it
    # unset/None and organelles are not measured at all. No default is
    # applied.
    'organelled_mask_dim': None,

    # (int) - (Depreceated) Minimum object area in square pixels. Most
    # classical segmenters and the U-Net discard smaller components during
    # segmentation via remove_small_objects (the LoG/DoG spot methods do
    # not, and the ring method applies a quarter of it, floor 3, to its
    # edge image), and the value is always applied again to the finished
    # label image. Despite the marker it is still live - raise it to clear
    # dim specks and hot pixels, lower it to keep faint puncta. Default
    # 10; 0 disables.
    'organelled_min_size': 0,

    # (int) - Position along the last axis of each merged/*.npy array
    # where the pathogen label mask sits, one plane after the nucleus
    # mask. With the default four image channels (0-3) that is 6; shift it
    # if you keep a different number of channels. None makes measure_crop
    # skip pathogen measurements, so infection status cannot be scored.
    # Default 6.
    'pathogen_mask_dim': 6,

    # (int) - (Depreceated) Minimum pathogen object area in pixels
    # squared, applied during measurement: any label with fewer pixels
    # than this is erased from the pathogen mask before features are
    # extracted. 0, the default, disables it. Superseded by
    # pathogen_min_area, which filters at segmentation time instead.
    'pathogen_min_size': 0,

    # (bool) - Render and save QC figures while the pipeline runs: channel
    # montages and Cellpose mask overlays during segmentation,
    # before/after filtration views and crop grids during measurement. It
    # adds figures per batch, so a full plate becomes much slower and more
    # memory-hungry; keep it for small or test_mode runs, which force it
    # on. Default False.
    'plot': False,

    # (dict) - Which source channel goes in each colour of the saved PNG,
    # e.g. {'r': 2, 'g': 1, 'b': 0}: channel 2 is red, 1 is green, 0 is
    # blue. Says outright what png_dims only implied. Channels not named
    # are absent from the crops (measurements are unaffected); a colour
    # left blank is an empty plane. Naming the same channel for all three
    # writes a greyscale PNG. Default {'r': 2, 'g': 1, 'b': 0}, which for
    # a standard 405/488/555 stack puts the nuclear stain in blue.
    'png_channel_mapping': {'r': 2, 'g': 1, 'b': 0},

    # (list of int) - Output crop size as [width, height] in pixels,
    # centred on the object centroid; larger keeps more surroundings,
    # smaller clips large objects. Should match the classifier input size
    # (default [224,224]). With several crop_mode entries pass a list of
    # lists, one size per mode, or a single size is reused for all.
    'png_size': [224, 224],

    # (bool) - Measure how each channel's intensity varies with distance
    # from the nucleus, pathogen and organelle boundaries inside each
    # cell, binned into 6 shells and saved as
    # <object>_rad_dist_channel_<c>_bin_0-5. Keep it on to quantify
    # recruitment or intensity gradients toward an object; turn it off to
    # shrink the feature table and speed up measurement. Default True.
    'radial_dist': True,

    # (bool) - Continue an interrupted run at its last verified safe
    # boundary instead of starting over. Each module verifies before it
    # accepts work as done: Mask revalidates existing mask and merged
    # arrays, Measure takes only fields complete in every table it owns
    # and clears partial rows before retrying, and the Format Converter
    # reopens each TIFF it checkpointed. So resuming cannot inherit a
    # half-written result -- the cost is the re-reading, not correctness.
    # Default False.
    'resume': False,

    # (bool) - Also save each object as a raw .npy array - all channels,
    # cropped to its bounding box, unnormalised - under a region_array/
    # folder. Enable when you need full bit depth or channels beyond
    # png_dims for custom analysis; it uses far more disk than PNGs.
    # Requires save_png to be True as well. Default False.
    'save_arrays': False,

    # (bool) - Master switch for the measurement half of measure_crop:
    # compute morphology and intensity features for every cell, nucleus,
    # pathogen, organelle and cytoplasm object and write them to the
    # plate's SQLite database. Set it False when you only want cropped
    # PNGs or filtered masks -- segmentation and cropping still run, but
    # no measurement tables are written. Default True.
    'save_measurements': True,

    # (bool) - Write one PNG crop per segmented object into
    # <crop_mode>_png/ and register each path in the png_list table of
    # measurements.db. Required for training or applying a classifier, for
    # the Annotate app and for the UMAP image plots. Turn off to only
    # compute measurements and save time and disk. Default True.
    'save_png': True,

    # (bool) - Measure each object's neighbourhood: how many neighbours
    # lie within a radius, the first and second nearest-neighbour
    # distances, and what fraction of its border touches another object.
    # Local density is the dominant confounder in image screens - crowded
    # cells are smaller, dimmer and cell-cycle-shifted - and without these
    # there is no per-object column to regress it out. Not produced for
    # cytoplasm, which is one object per cell by construction. Costs one
    # KD-tree and one boundary pass per field, not per object. Default
    # False.
    'spatial_measurements': False,

    # (str, path) - Folder the current step reads from and writes into:
    # raw images for mask generation, the merged/ folder of .npy stacks
    # for measure, the plate root for dataset/regression steps, or the
    # folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/,
    # measurements/measurements.db, datasets/, results/) are created
    # inside it. A list of paths, or a "['a','b']" string, processes
    # several plates in one run. No usable default: the settings factories
    # fill a placeholder ('path' or '/path/to/src'), so this must be
    # supplied.
    'src': 'path',

    # (bool or None) - What happens when a step hits a problem it could
    # survive. OFF records the failure in the run ledger and the end-of-
    # run summary and carries on with the items that worked. ON raises
    # immediately on a setup or configuration error -- an unreadable path,
    # a missing column, a database that will not open -- so a batch stops
    # at the first sign its inputs are wrong instead of producing a
    # plausible partial result. Per-item failures such as one corrupt
    # image are survived either way. None defers to $SPACR_STRICT_ERRORS,
    # which is how a cluster sets it for a whole batch. Default None.
    'strict_errors': None,

    # (str, list or None) - Parent compartments to roll every enabled
    # organelle slot into. Accepts 'cell', 'nucleus', 'pathogen' and
    # 'cytoplasm'; each writes one <parent>_organelle_summary row per
    # parent with a separate organelle_summary_<slot>_* column family. Raw
    # per-organelle tables are always written when their mask dim is
    # enabled. Default 'cell'; None disables only these rollups.
    'summarize_organelles_by': 'cell',

    # (bool) - Run the pipeline on a small random subset instead of the
    # whole folder. Mask generation copies test_images (default 10)
    # complete image sets into <src>/test and works there; measure_crop
    # copies test_nr (default 10) merged arrays into test/merged. Both
    # also force verbose and plot on. Use it to check channel assignment,
    # diameters and thresholds before committing to a full plate. Default
    # False.
    'test_mode': False,

    # (int) - How many files are sampled at random from merged/ into
    # test/merged when test_mode is on in the measure-and-crop pipeline,
    # so measurement runs on a small subset. Raise it if a handful of
    # fields is not representative; each extra file costs a full
    # measurement pass. Default 10.
    'test_nr': 10,

    # (bool) - Treat each well/field as a time series instead of
    # independent images: files are grouped into time stacks,
    # randomization is switched off, per-channel movies are written,
    # objects in timelapse_objects are tracked across frames, a timeID
    # column is added to the measurement tables, and measure_crop stops
    # writing single-object PNGs. Only enable when filenames carry a time
    # index. Default False.
    'timelapse': False,

    # (list) - Which segmented objects are tracked across frames and
    # relabelled with track IDs: any subset of ['cell', 'nucleus',
    # 'pathogen']; any other value aborts the run with a message. Each
    # extra entry costs a full additional tracking pass. Tracking nuclei
    # is often more stable than cells when cells touch. Default ['cell'].
    'timelapse_objects': ['cell'],

    # (bool) - Decides which cells survive the consistency filter in
    # measure_crop. True keeps any cell that has both a nucleus and a
    # cytoplasm; False also demands at least one pathogen, dropping
    # uninfected cells from every table. Either way,
    # nucleus/pathogen/cytoplasm labels outside the surviving cells are
    # zeroed. Only applied when cell, nucleus and pathogen masks all
    # exist; forced True otherwise. Default True.
    'uninfected': True,

    # (bool) - Crop the object's rectangular bounding box padded by 10 px
    # instead of its mask, so neighbouring cells and background inside the
    # box are kept rather than zeroed out. Enable when the classifier
    # should see local context; leave off to isolate a single object on a
    # black background. Default False.
    'use_bounding_box': False,

    # (bool) - Print extra run detail instead of the minimal log: the
    # resolved settings table, the channel and model choices per object
    # type, per-table row counts, and how many objects survive each
    # filter. It only adds console output, so turn it on when object
    # counts come out unexpected and you need to see which stage removed
    # them. The default differs per pipeline -- True for mask, UMAP,
    # screen analysis, barcode mapping and Cellpose training; False for
    # measure, the plotting helpers and regression.
    'verbose': False,

    # (float or None) - Width of one pixel in micrometres in the image
    # plane, assumed square. Used with voxel_size_z_um to derive
    # anisotropy and to turn voxel counts into physical volumes and
    # surface areas. Note this is a different setting from um_per_pixel,
    # which only sizes the scale bar drawn on figures and never reaches a
    # measurement. This one does reach measurements, but only on a 3-D
    # run: a 2-D run never applies it, because doing so would turn every
    # *_area from px2 into um2 under an unchanged column name. Default
    # None.
    'voxel_size_xy_um': None,

    # (float or None) - Spacing between consecutive z planes in
    # micrometres, straight off the acquisition settings. Together with
    # voxel_size_xy_um it derives anisotropy, so setting these two is the
    # safer way to get it right, and it is also what converts object
    # volumes from voxel counts into um3. Changing it rescales every
    # physical z quantity and the anisotropy used for segmentation; it has
    # no effect on a 'project' run. Measure uses the pair to report 3-D
    # morphology in micrometres rather than voxels, and records which it
    # used in the measurement_units column. Default None.
    'voxel_size_z_um': None,
}

## 4. Run it

This is the long cell. Progress is logged; if you want more of it, raise the log levels in Preferences → Logging, or set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter.

In [ ]:
measure_crop(settings)

## Where the output went

`measurements/measurements.db` and, when cropping is on, one image per object under `data/`.

spaCR writes beside the source folder rather than into a global location, so a plate stays self-contained and re-running does not clobber a different experiment.

### Next steps

* The GUI covers the same workflows with the settings laid out as a form — `python -m spacr`.
* The narrated walkthroughs are at <https://einarolafsson.github.io/spacr/tutorials/>.
* The API reference is at <https://einarolafsson.github.io/spacr/>.